**4 quy tắc lọc chống trùng lặp theo thời gian**:
1. **Quy tắc 1 (Category)**: Dữ liệu danh mục chỉ nạp 1 lần duy nhất, các lần sau tự động bỏ qua.
2. **Quy tắc 2 (Daily Price)**: So khóa từng dòng (`product_id, seller_product_id, crawl_date`) với database; chỉ thêm khóa chưa có, kể cả dữ liệu mới trong cùng ngày.
3. **Quy tắc 3 (Monthly Tables)**: So khóa từng dòng snapshot với database; tháng cũ vẫn được rà để lấy các khóa mới còn thiếu.
4. **Quy tắc 4 (Reviews)**: Lọc theo khóa chính `review_id, product_id, seller_product_id` (seller_id có thể rỗng), chỉ nạp các đánh giá mới chưa từng xuất hiện trong database.

Tất cả trạng thái nạp Supabase được ghi nhận vào file: `03_push_to_sql/sync_checkpoint_supabase.json`.

In [ ]:
from pathlib import Path
import os
import psycopg2
import json
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = next((p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "02_process_raw_data").is_dir()), CURRENT_DIR)

# Ưu tiên biến môi trường; local có thể cấu hình SUPABASE_DB_URL trong file .env (đã được .gitignore).
DB_URL = os.environ.get("SUPABASE_DB_URL")
if not DB_URL and (PROJECT_DIR / ".env").exists():
    for line in (PROJECT_DIR / ".env").read_text(encoding="utf-8").splitlines():
        key, separator, value = line.partition("=")
        if separator and key.strip() == "SUPABASE_DB_URL":
            DB_URL = value.strip().strip("\"'")
            break
if not DB_URL:
    raise RuntimeError("Thiếu SUPABASE_DB_URL: cấu hình biến môi trường hoặc file .env")
CHECKPOINT_FILE = PROJECT_DIR / "03_push_to_sql" / "sync_checkpoint_supabase.json"
LOG_FILE = PROJECT_DIR / "03_push_to_sql" / "logs" / "notebook_incremental.log"

print(f"Supabase Database Connected")
print(f"Checkpoint file: {CHECKPOINT_FILE}")

## 1. Xem file Checkpoint hiện tại (Những gì đã lưu)

In [ ]:
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        state = json.load(f)
    print(json.dumps(state, indent=2, ensure_ascii=False))
else:
    print("Chưa có file checkpoint. Sẽ được tự động tạo khi chạy nạp.")

## 2. Chạy tiến trình nạp gia tăng thông minh (Incremental Sync)

In [ ]:
from push_to_database import push_incremental_postgres

# Chạy kiểm tra: các dữ liệu đã nạp sẽ tự động [BỎ QUA], không bị nạp trùng lên Cloud
push_incremental_postgres(PROJECT_DIR, DB_URL, LOG_FILE)

## 3. Thống kê tổng quan dữ liệu trong Database

In [ ]:
conn = psycopg2.connect(DB_URL)
tables = [
    "dim_category", "dim_sub_category", "dim_product_listing",
    "product_detail", "product_seller", "product_category_map",
    "fact_daily_price", "fact_review"
]

rows = []
for t in tables:
    df_count = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {t};", conn)
    c = df_count["total"].iloc[0]
    rows.append({"Bảng": t, "Tổng số dòng": f"{c:,}"})

display(pd.DataFrame(rows))
conn.close()